# IPL Crunch '26 Analysis

## Dataset Extraction and Parsing
Converting IPL JSON match files into structured match-level and ball-by-ball datasets.

In [ ]:
# Inspect the README first and preview its contents so I can understand dataset structure before parsing files.
import zipfile
from pathlib import Path

zip_path = Path('ipl_json.zip')
with zipfile.ZipFile(zip_path, 'r') as zip_file:
    readme_text = zip_file.read('README.txt').decode('utf-8', errors='ignore')

print(readme_text[:5000])

In [ ]:
# Extract and parse the IPL JSON archive into match and ball-level tables, then show quick previews for verification.
import zipfile, json, pandas as pd
from tqdm import tqdm

match_rows = []
delivery_rows = []

with zipfile.ZipFile('ipl_json.zip', 'r') as zip_file:
    json_names = [nm for nm in zip_file.namelist() if nm.endswith('.json') and nm != 'README.txt']
    for json_name in tqdm(json_names):
        match_data = json.loads(zip_file.read(json_name))
        info_data = match_data.get('info', {})
        match_id = json_name.replace('.json', '')
        dates_list = info_data.get('dates', [])
        event_data = info_data.get('event', {}) if isinstance(info_data.get('event', {}), dict) else {}
        outcome_data = info_data.get('outcome', {}) if isinstance(info_data.get('outcome', {}), dict) else {}
        toss_data = info_data.get('toss', {}) if isinstance(info_data.get('toss', {}), dict) else {}
        registry_people = (((info_data.get('registry') or {}).get('people')) or {})
        teams_list = info_data.get('teams', [None, None])
        winners_list = outcome_data.get('winner')
        win_by = None
        win_margin = None
        if 'by' in outcome_data and isinstance(outcome_data['by'], dict):
            by_data = outcome_data['by']
            if 'runs' in by_data:
                win_by = 'runs'
                win_margin = by_data.get('runs')
            elif 'wickets' in by_data:
                win_by = 'wickets'
                win_margin = by_data.get('wickets')
        match_rows.append({
            'match_id': match_id,
            'season': info_data.get('season'),
            'date': dates_list[0] if len(dates_list) > 0 else None,
            'team1': teams_list[0] if len(teams_list) > 0 else None,
            'team2': teams_list[1] if len(teams_list) > 1 else None,
            'venue': info_data.get('venue'),
            'city': info_data.get('city'),
            'toss_winner': toss_data.get('winner'),
            'toss_decision': toss_data.get('decision'),
            'winner': winners_list,
            'win_by': win_by,
            'win_margin': win_margin,
            'result': outcome_data.get('result'),
            'method': outcome_data.get('method'),
            'player_of_match': ', '.join(info_data.get('player_of_match', [])) if info_data.get('player_of_match') else None,
            'event_name': event_data.get('name'),
            'match_number': event_data.get('match_number'),
            'gender': info_data.get('gender'),
            'team_type': info_data.get('team_type')
        })
        innings_list = match_data.get('innings', [])
        for innings_idx, innings_obj in enumerate(innings_list, start=1):
            innings_name = list(innings_obj.keys())[0]
            innings_data = innings_obj[innings_name]
            batting_team = innings_data.get('team')
            for over_obj in innings_data.get('overs', []):
                over_num = over_obj.get('over')
                for ball_obj in over_obj.get('deliveries', []):
                    batter_runs = ((ball_obj.get('runs') or {}).get('batter')) or 0
                    extra_runs = ((ball_obj.get('runs') or {}).get('extras')) or 0
                    total_runs = ((ball_obj.get('runs') or {}).get('total')) or 0
                    wicket_list = ball_obj.get('wickets', []) or []
                    delivery_rows.append({
                        'match_id': match_id,
                        'season': info_data.get('season'),
                        'date': dates_list[0] if len(dates_list) > 0 else None,
                        'innings': innings_idx,
                        'innings_name': innings_name,
                        'batting_team': batting_team,
                        'bowling_team': teams_list[0] if batting_team == (teams_list[1] if len(teams_list) > 1 else None) else (teams_list[1] if len(teams_list) > 1 else None),
                        'over': over_num,
                        'ball_in_over': ball_obj.get('ball'),
                        'batter': ball_obj.get('batter'),
                        'bowler': ball_obj.get('bowler'),
                        'non_striker': ball_obj.get('non_striker'),
                        'batter_runs': batter_runs,
                        'extras': extra_runs,
                        'total_runs': total_runs,
                        'is_boundary_4': 1 if batter_runs == 4 else 0,
                        'is_boundary_6': 1 if batter_runs == 6 else 0,
                        'is_dot': 1 if total_runs == 0 else 0,
                        'wicket_on_ball': 1 if len(wicket_list) > 0 else 0,
                        'dismissed_player': wicket_list[0].get('player_out') if len(wicket_list) > 0 else None,
                        'dismissal_kind': wicket_list[0].get('kind') if len(wicket_list) > 0 else None
                    })

matches_df = pd.DataFrame(match_rows)
deliveries_df = pd.DataFrame(delivery_rows)
matches_df['date'] = pd.to_datetime(matches_df['date'], errors='coerce')
deliveries_df['date'] = pd.to_datetime(deliveries_df['date'], errors='coerce')
deliveries_df['over'] = pd.to_numeric(deliveries_df['over'], errors='coerce')

print(matches_df.head())
print(deliveries_df.head())

## Data Cleaning and DataFrame Creation
Processing innings structure and preparing reusable analysis tables.

In [ ]:
# Inspect a few raw JSON objects to adapt the parser to the actual innings structure used in this archive.
import zipfile, json
from itertools import islice

with zipfile.ZipFile('ipl_json.zip', 'r') as zip_file:
    json_names = [nm for nm in zip_file.namelist() if nm.endswith('.json') and nm != 'README.txt']
    for json_name in list(islice(json_names, 0, 2)):
        match_data = json.loads(zip_file.read(json_name))
        print(json_name)
        print(type(match_data.get('innings')))
        print(match_data.get('innings')[0])
        print('---')

In [ ]:
# Parse the archive with the correct innings structure, create reusable dataframes, and show heads immediately for validation.
import zipfile, json, pandas as pd
from tqdm import tqdm

match_rows = []
delivery_rows = []

with zipfile.ZipFile('ipl_json.zip', 'r') as zip_file:
    json_names = [nm for nm in zip_file.namelist() if nm.endswith('.json') and nm != 'README.txt']
    for json_name in tqdm(json_names):
        match_data = json.loads(zip_file.read(json_name))
        info_data = match_data.get('info', {})
        match_id = json_name.replace('.json', '')
        dates_list = info_data.get('dates', [])
        event_data = info_data.get('event', {}) if isinstance(info_data.get('event', {}), dict) else {}
        outcome_data = info_data.get('outcome', {}) if isinstance(info_data.get('outcome', {}), dict) else {}
        toss_data = info_data.get('toss', {}) if isinstance(info_data.get('toss', {}), dict) else {}
        teams_list = info_data.get('teams', [None, None])
        winner_name = outcome_data.get('winner')
        win_by = None
        win_margin = None
        if isinstance(outcome_data.get('by'), dict):
            if 'runs' in outcome_data['by']:
                win_by = 'runs'
                win_margin = outcome_data['by'].get('runs')
            elif 'wickets' in outcome_data['by']:
                win_by = 'wickets'
                win_margin = outcome_data['by'].get('wickets')
        match_rows.append({
            'match_id': match_id,
            'season': str(info_data.get('season')),
            'date': dates_list[0] if len(dates_list) > 0 else None,
            'team1': teams_list[0] if len(teams_list) > 0 else None,
            'team2': teams_list[1] if len(teams_list) > 1 else None,
            'venue': info_data.get('venue'),
            'city': info_data.get('city'),
            'toss_winner': toss_data.get('winner'),
            'toss_decision': toss_data.get('decision'),
            'winner': winner_name,
            'win_by': win_by,
            'win_margin': win_margin,
            'result': outcome_data.get('result'),
            'method': outcome_data.get('method'),
            'player_of_match': ', '.join(info_data.get('player_of_match', [])) if info_data.get('player_of_match') else None,
            'event_name': event_data.get('name'),
            'match_number': event_data.get('match_number')
        })
        innings_list = match_data.get('innings', [])
        for innings_idx, innings_data in enumerate(innings_list, start=1):
            batting_team = innings_data.get('team')
            team_a = teams_list[0] if len(teams_list) > 0 else None
            team_b = teams_list[1] if len(teams_list) > 1 else None
            bowling_team = team_b if batting_team == team_a else team_a
            for over_obj in innings_data.get('overs', []):
                over_num = over_obj.get('over')
                for ball_pos, ball_obj in enumerate(over_obj.get('deliveries', []), start=1):
                    runs_data = ball_obj.get('runs', {}) or {}
                    wicket_list = ball_obj.get('wickets', []) or []
                    delivery_rows.append({
                        'match_id': match_id,
                        'season': str(info_data.get('season')),
                        'date': dates_list[0] if len(dates_list) > 0 else None,
                        'innings': innings_idx,
                        'batting_team': batting_team,
                        'bowling_team': bowling_team,
                        'over': over_num,
                        'ball_in_over': ball_pos,
                        'batter': ball_obj.get('batter'),
                        'bowler': ball_obj.get('bowler'),
                        'non_striker': ball_obj.get('non_striker'),
                        'batter_runs': runs_data.get('batter', 0),
                        'extras': runs_data.get('extras', 0),
                        'total_runs': runs_data.get('total', 0),
                        'is_boundary_4': 1 if runs_data.get('batter', 0) == 4 else 0,
                        'is_boundary_6': 1 if runs_data.get('batter', 0) == 6 else 0,
                        'is_dot': 1 if runs_data.get('total', 0) == 0 else 0,
                        'wicket_on_ball': 1 if len(wicket_list) > 0 else 0,
                        'dismissed_player': wicket_list[0].get('player_out') if len(wicket_list) > 0 else None,
                        'dismissal_kind': wicket_list[0].get('kind') if len(wicket_list) > 0 else None
                    })

matches_df = pd.DataFrame(match_rows)
deliveries_df = pd.DataFrame(delivery_rows)
matches_df['date'] = pd.to_datetime(matches_df['date'], errors='coerce')
deliveries_df['date'] = pd.to_datetime(deliveries_df['date'], errors='coerce')

print(matches_df.head())
print(deliveries_df.head())
print(matches_df.shape)
print(deliveries_df.shape)

## Toss Impact Analysis
Analyzing whether winning the toss significantly affects match outcomes.

In [ ]:
# Build phase, toss, batter, and bowler summaries plus a surprise-insight table for a concise competition-ready analysis.
import pandas as pd

valid_matches_df = matches_df[matches_df['winner'].notna()].copy()
valid_matches_df['toss_win_match_win'] = (valid_matches_df['toss_winner'] == valid_matches_df['winner']).astype(int)

deliveries_df['phase'] = pd.cut(
    deliveries_df['over'],
    bins=[-1, 5, 14, 20],
    labels=['Powerplay', 'Middle Overs', 'Death Overs']
)

phase_team_df = deliveries_df.groupby(['match_id', 'innings', 'batting_team', 'phase'], as_index=False).agg(
    runs=('total_runs', 'sum'),
    wickets_lost=('wicket_on_ball', 'sum'),
    balls=('total_runs', 'size'),
    boundaries_4=('is_boundary_4', 'sum'),
    boundaries_6=('is_boundary_6', 'sum')
)
phase_team_df['run_rate'] = phase_team_df['runs'] / phase_team_df['balls'] * 6

innings_team_df = deliveries_df.groupby(['match_id', 'innings', 'batting_team'], as_index=False).agg(total_runs=('total_runs', 'sum'))
match_winner_map = valid_matches_df[['match_id', 'winner']].copy()
innings_team_df = innings_team_df.merge(match_winner_map, on='match_id', how='left')
innings_team_df['won_match'] = (innings_team_df['batting_team'] == innings_team_df['winner']).astype(int)
phase_team_df = phase_team_df.merge(innings_team_df[['match_id', 'innings', 'batting_team', 'won_match']], on=['match_id', 'innings', 'batting_team'], how='left')

phase_impact_df = phase_team_df.groupby('phase', as_index=False).agg(
    avg_run_rate=('run_rate', 'mean'),
    avg_wickets=('wickets_lost', 'mean'),
    win_run_rate=('run_rate', lambda vals: phase_team_df.loc[vals.index, 'won_match'].corr(vals)),
    win_wickets=('wickets_lost', lambda vals: phase_team_df.loc[vals.index, 'won_match'].corr(vals))
)

batter_season_df = deliveries_df.groupby(['season', 'batter'], as_index=False).agg(
    runs=('batter_runs', 'sum'),
    balls=('batter_runs', 'size'),
    fours=('is_boundary_4', 'sum'),
    sixes=('is_boundary_6', 'sum'),
    dismissals=('wicket_on_ball', 'sum')
)
batter_season_df['strike_rate'] = batter_season_df['runs'] / batter_season_df['balls'] * 100
batter_season_df['boundary_runs_pct'] = ((batter_season_df['fours'] * 4) + (batter_season_df['sixes'] * 6)) / batter_season_df['runs'].replace(0, pd.NA)

top_batters_df = batter_season_df[batter_season_df['balls'] >= 150].sort_values(['runs', 'strike_rate'], ascending=[False, False]).groupby('season').head(5).copy()

bowler_season_df = deliveries_df.groupby(['season', 'bowler'], as_index=False).agg(
    wickets=('wicket_on_ball', 'sum'),
    balls=('total_runs', 'size'),
    runs_conceded=('total_runs', 'sum'),
    dots=('is_dot', 'sum')
)
bowler_season_df['economy'] = bowler_season_df['runs_conceded'] / bowler_season_df['balls'] * 6
bowler_season_df['strike_rate'] = bowler_season_df['balls'] / bowler_season_df['wickets'].replace(0, pd.NA)

top_bowlers_df = bowler_season_df[bowler_season_df['balls'] >= 120].sort_values(['wickets', 'economy'], ascending=[False, True]).groupby('season').head(5).copy()

team_win_rate_df = pd.concat([
    valid_matches_df[['season', 'team1', 'winner']].rename(columns={'team1': 'team'}),
    valid_matches_df[['season', 'team2', 'winner']].rename(columns={'team2': 'team'})
], ignore_index=True)
team_win_rate_df['won'] = (team_win_rate_df['team'] == team_win_rate_df['winner']).astype(int)
team_win_summary_df = team_win_rate_df.groupby(['season', 'team'], as_index=False).agg(matches=('won', 'size'), wins=('won', 'sum'))
team_win_summary_df['win_pct'] = team_win_summary_df['wins'] / team_win_summary_df['matches']

print(valid_matches_df[['match_id', 'season', 'team1', 'team2', 'toss_winner', 'winner', 'toss_win_match_win']].head())
print(phase_impact_df)
print(top_batters_df.head())
print(top_bowlers_df.head())

## Match Phase Impact

Comparing the impact of Powerplay, Middle Overs, and Death Overs.

In [ ]:
# Recompute the summaries with safer groupby logic, then print compact tables needed for charts and conclusions.
import pandas as pd

valid_matches_df = matches_df[matches_df['winner'].notna()].copy()
valid_matches_df['toss_win_match_win'] = (valid_matches_df['toss_winner'] == valid_matches_df['winner']).astype(int)

if 'phase' not in deliveries_df.columns:
    deliveries_df['phase'] = pd.cut(deliveries_df['over'], bins=[-1, 5, 14, 20], labels=['Powerplay', 'Middle Overs', 'Death Overs'])

phase_team_df = deliveries_df.groupby(['match_id', 'innings', 'batting_team', 'phase'], observed=True).agg(
    runs=('total_runs', 'sum'),
    wickets_lost=('wicket_on_ball', 'sum'),
    balls=('total_runs', 'size')
).reset_index()
phase_team_df['run_rate'] = phase_team_df['runs'] / phase_team_df['balls'] * 6

winner_map_df = valid_matches_df[['match_id', 'winner']].drop_duplicates()
phase_team_df = phase_team_df.merge(winner_map_df, on='match_id', how='left')
phase_team_df['won_match'] = (phase_team_df['batting_team'] == phase_team_df['winner']).astype(int)

phase_impact_df = phase_team_df.groupby('phase', observed=True).apply(
    lambda part_df: pd.Series({
        'avg_run_rate': part_df['run_rate'].mean(),
        'avg_wickets': part_df['wickets_lost'].mean(),
        'corr_run_rate_win': part_df['run_rate'].corr(part_df['won_match']),
        'corr_wickets_win': part_df['wickets_lost'].corr(part_df['won_match'])
    })
).reset_index()

toss_summary_df = valid_matches_df.groupby('season', as_index=False).agg(
    matches=('match_id', 'nunique'),
    toss_and_match_wins=('toss_win_match_win', 'sum')
)
toss_summary_df['toss_win_match_win_pct'] = toss_summary_df['toss_and_match_wins'] / toss_summary_df['matches']

top_batters_df = deliveries_df.groupby(['season', 'batter']).agg(
    runs=('batter_runs', 'sum'),
    balls=('batter_runs', 'size'),
    fours=('is_boundary_4', 'sum'),
    sixes=('is_boundary_6', 'sum')
).reset_index()
top_batters_df = top_batters_df[top_batters_df['balls'] >= 150].copy()
top_batters_df['strike_rate'] = top_batters_df['runs'] / top_batters_df['balls'] * 100
top_batters_df = top_batters_df.sort_values(['season', 'runs', 'strike_rate'], ascending=[True, False, False])
season_top_batters_df = top_batters_df.groupby('season').head(5).copy()

bowler_summary_df = deliveries_df.groupby(['season', 'bowler']).agg(
    wickets=('wicket_on_ball', 'sum'),
    balls=('total_runs', 'size'),
    runs_conceded=('total_runs', 'sum'),
    dots=('is_dot', 'sum')
).reset_index()
bowler_summary_df = bowler_summary_df[bowler_summary_df['balls'] >= 120].copy()
bowler_summary_df['economy'] = bowler_summary_df['runs_conceded'] / bowler_summary_df['balls'] * 6
season_top_bowlers_df = bowler_summary_df.sort_values(['season', 'wickets', 'economy'], ascending=[True, False, True]).groupby('season').head(5).copy()

print(toss_summary_df.head())
print(phase_impact_df)
print(season_top_batters_df.head())
print(season_top_bowlers_df.head())

## Top Batters and Bowlers

Identifying the highest-performing IPL players across seasons.

In [ ]:
# Compute polished summary tables and create a few competition-ready charts from the already loaded IPL dataframes.
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style='whitegrid')

overall_toss_pct = valid_matches_df['toss_win_match_win'].mean()
phase_rank_df = phase_impact_df.sort_values('corr_run_rate_win', ascending=False).copy()

overall_batter_df = deliveries_df.groupby('batter').agg(
    runs=('batter_runs', 'sum'),
    balls=('batter_runs', 'size'),
    fours=('is_boundary_4', 'sum'),
    sixes=('is_boundary_6', 'sum')
).reset_index()
overall_batter_df = overall_batter_df[overall_batter_df['balls'] >= 500].copy()
overall_batter_df['strike_rate'] = overall_batter_df['runs'] / overall_batter_df['balls'] * 100
overall_batter_df = overall_batter_df.sort_values(['runs', 'strike_rate'], ascending=[False, False]).head(10)

overall_bowler_df = deliveries_df.groupby('bowler').agg(
    wickets=('wicket_on_ball', 'sum'),
    balls=('total_runs', 'size'),
    runs_conceded=('total_runs', 'sum')
).reset_index()
overall_bowler_df = overall_bowler_df[overall_bowler_df['balls'] >= 500].copy()
overall_bowler_df['economy'] = overall_bowler_df['runs_conceded'] / overall_bowler_df['balls'] * 6
overall_bowler_df = overall_bowler_df.sort_values(['wickets', 'economy'], ascending=[False, True]).head(10)

team_match_df = pd.concat([
    valid_matches_df[['match_id', 'season', 'team1', 'winner']].rename(columns={'team1': 'team'}),
    valid_matches_df[['match_id', 'season', 'team2', 'winner']].rename(columns={'team2': 'team'})
], ignore_index=True)
team_match_df['won'] = (team_match_df['team'] == team_match_df['winner']).astype(int)
team_all_time_df = team_match_df.groupby('team').agg(matches=('match_id', 'nunique'), wins=('won', 'sum')).reset_index()
team_all_time_df = team_all_time_df[team_all_time_df['matches'] >= 30].copy()
team_all_time_df['win_pct'] = team_all_time_df['wins'] / team_all_time_df['matches'] * 100
team_all_time_df = team_all_time_df.sort_values('win_pct', ascending=False)

print(round(overall_toss_pct * 100, 2))
print(phase_rank_df)
print(overall_batter_df.head())
print(overall_bowler_df.head())
print(team_all_time_df.head())

plt.figure(figsize=(8,4))
plot_toss = toss_summary_df.sort_values('season').copy()
plot_toss['pct'] = plot_toss['toss_win_match_win_pct'] * 100
sns.lineplot(data=plot_toss, x='season', y='pct', marker='o', color='#1f77b4')
plt.axhline(overall_toss_pct * 100, linestyle='--', color='#d62728')
plt.xticks(rotation=60)
plt.ylabel('Percent')
plt.xlabel('Season')
plt.title('How often toss winners also won the match')
plt.tight_layout()
plt.show()

plt.figure(figsize=(7,4))
sns.barplot(data=phase_rank_df, x='phase', y='corr_run_rate_win', palette=['#ef553b','#636efa','#00cc96'])
plt.ylabel('Correlation with winning')
plt.xlabel('Phase')
plt.title('Which batting phase tracks winning the most')
plt.tight_layout()
plt.show()

plt.figure(figsize=(9,5))
sns.barplot(data=overall_batter_df, y='batter', x='runs', color='#ff7f0e')
plt.xlabel('Career IPL runs in dataset')
plt.ylabel('Batter')
plt.title('Top batters across all seasons')
plt.tight_layout()
plt.show()

plt.figure(figsize=(9,5))
sns.barplot(data=overall_bowler_df, y='bowler', x='wickets', color='#2ca02c')
plt.xlabel('Career IPL wickets in dataset')
plt.ylabel('Bowler')
plt.title('Top bowlers across all seasons')
plt.tight_layout()
plt.show()

## Exporting Analysis Outputs

Saving summary tables for reporting and presentation.

In [ ]:
# Save the main summary tables to csv files so the user can reuse them in a report or dashboard submission.
summary_toss_export_df = toss_summary_df.copy()
summary_phase_export_df = phase_impact_df.copy()
summary_batter_export_df = overall_batter_df.copy()
summary_bowler_export_df = overall_bowler_df.copy()
summary_team_export_df = team_all_time_df.copy()

summary_toss_export_df.to_csv('ipl_toss_summary.csv', index=False)
summary_phase_export_df.to_csv('ipl_phase_impact_summary.csv', index=False)
summary_batter_export_df.to_csv('ipl_top_batters_all_time.csv', index=False)
summary_bowler_export_df.to_csv('ipl_top_bowlers_all_time.csv', index=False)
summary_team_export_df.to_csv('ipl_team_win_rates.csv', index=False)

print('ipl_toss_summary.csv')
print('ipl_phase_impact_summary.csv')
print('ipl_top_batters_all_time.csv')
print('ipl_top_bowlers_all_time.csv')
print('ipl_team_win_rates.csv')

In [ ]:
# Build a compact competition-ready insight pack from the IPL data already in memory and export a notebook-friendly HTML summary.
import pandas as pd
from IPython.display import HTML, display

winner_only_df = matches_df[matches_df['winner'].notna()].copy()
winner_only_df['toss_match_win'] = (winner_only_df['toss_winner'] == winner_only_df['winner']).astype(int)

phase_compare_df = phase_team_df.groupby(['phase', 'won_match'], observed=True).agg(
    avg_run_rate=('run_rate', 'mean'),
    avg_wickets=('wickets_lost', 'mean')
).reset_index()
phase_compare_df['result'] = phase_compare_df['won_match'].map({1: 'Winners', 0: 'Losers'})

venue_chase_df = winner_only_df.copy()
venue_chase_df['chasing_team_won'] = ((venue_chase_df['toss_decision'] == 'field') & (venue_chase_df['winner'] == venue_chase_df['toss_winner'])) | ((venue_chase_df['toss_decision'] == 'bat') & (venue_chase_df['winner'] != venue_chase_df['toss_winner']))
venue_chase_df = venue_chase_df.groupby('venue').agg(matches=('match_id', 'count'), chase_wins=('chasing_team_won', 'sum')).reset_index()
venue_chase_df = venue_chase_df[venue_chase_df['matches'] >= 10].copy()
venue_chase_df['chase_win_pct'] = venue_chase_df['chase_wins'] / venue_chase_df['matches'] * 100
venue_chase_df = venue_chase_df.sort_values('chase_win_pct', ascending=False).head(10)

surprise_text = 'Winning the toss only translated into winning the match ' + str(round(winner_only_df['toss_match_win'].mean() * 100, 1)) + '% of the time, which is much smaller than the usual fan narrative.'

html_string = '<div style="font-family:Arial,sans-serif;padding:18px;line-height:1.45">'
html_string += '<h2 style="margin-bottom:6px">IPL Crunch 26 - Competition Ready Notebook Summary</h2>'
html_string += '<p><b>Dataset:</b> 1,226 matches and 291,574 deliveries parsed from Cricsheet JSON.</p>'
html_string += '<h3>Core findings</h3>'
html_string += '<ul>'
html_string += '<li><b>Toss effect:</b> Toss winners also won the match in <b>' + str(round(winner_only_df['toss_match_win'].mean() * 100, 2)) + '%</b> of games.</li>'
html_string += '<li><b>Most important phase:</b> <b>Death overs</b> had the strongest relationship with winning.</li>'
html_string += '<li><b>Top all-time batter:</b> <b>' + str(overall_batter_df.iloc[0]['batter']) + '</b> with <b>' + str(int(overall_batter_df.iloc[0]['runs'])) + '</b> runs.</li>'
html_string += '<li><b>Top all-time bowler:</b> <b>' + str(overall_bowler_df.iloc[0]['bowler']) + '</b> with <b>' + str(int(overall_bowler_df.iloc[0]['wickets'])) + '</b> wickets.</li>'
html_string += '</ul>'
html_string += '<h3>One insight that genuinely surprised me</h3>'
html_string += '<p>' + surprise_text + '</p>'
html_string += '<h3>Best venues for chasing</h3>' + venue_chase_df.head(5).to_html(index=False)
html_string += '<h3>Winners vs losers by phase</h3>' + phase_compare_df.to_html(index=False)
html_string += '</div>'

display(HTML(html_string))

with open('ipl_competition_ready_summary.html', 'w', encoding='utf-8') as file_obj:
    file_obj.write(html_string)

print('ipl_competition_ready_summary.html')

# Key Insights and Conclusion

- Toss advantage exists but is relatively small.
- Death overs have the strongest relationship with winning.
- Long-term consistency defines elite IPL players.
- Data-driven analysis reveals patterns beyond fan intuition.